In [2]:
import os 
import math 
from typing import  Literal ,Dict,List
from typing_extensions import  TypedDict
from pydantic import  BaseModel,Field
from langgraph.graph import  StateGraph,START,END,MessagesState 
from langchain_core.messages import  HumanMessage,BaseMessage
from langgraph.prebuilt import  ToolNode
from langchain_groq  import ChatGroq
from langchain_core.tools import  tool
from dotenv import  load_dotenv



In [3]:
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")

In [5]:
class Clinical_state(MessagesState):
    patient_vitals:Dict[str:Any]
    risk_score:float
    next_node:str

In [7]:
class Superviser(MessagesState):
    """the superviser manage all the agents and work in the cordination also in the best loop """
    next_node:Literal["validation_agent", "cardio_ml_worker", "reporting_worker", "FINISH"]=Field(
        discription="Route to validation if vitals are missing, cardio if data is ready, reporting if risk is calculated.")


C:\Users\Malik\AppData\Local\Temp\ipykernel_18220\4189312931.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'discription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  next_node:Literal["validation_agent", "cardio_ml_worker", "reporting_worker", "FINISH"]=Field(


In [11]:
@tool
def vec_db(patient:str)->str:
    """Search vector database for historical patient vitals."""
    rec={
        "khan": "Historical vitals: age 55, previous cholesterol 245, max heart rate 140.",
        "azmat": "Historical vitals: age 42, previous cholesterol 190, max heart rate 165."
    }
    
    return rec.get(patient,"no rec found")

@tool
def search_clinical_literature(symptom: str) -> str:
    """Query external literature for symptom risk factors."""
    if "chest pain" in symptom.lower():
        return "Literature indicates atypical angina combined with cholesterol > 240 increases risk."
    return "No significant recent literature flags for this symptom."


    
tools=[vec_db,search_clinical_literature]

t_llm=llm.bind_tools(tools)


In [ ]:

# ==========================================
# 2. REACT TOOLS (PHASE 1)
# ==========================================
@tool
def search_ehr_vector_db(patient_name: str) -> str:
    """Search vector database for historical patient vitals."""
    # Fully expanded mock DB retrieval
    db_records = {
        "John Doe": "Historical vitals: age 55, previous cholesterol 245, max heart rate 140.",
        "Jane Smith": "Historical vitals: age 42, previous cholesterol 190, max heart rate 165."
    }
    return db_records.get(patient_name, "No historical EHR data found.")

@tool
def search_clinical_literature(symptom: str) -> str:
    """Query external literature for symptom risk factors."""
    if "chest pain" in symptom.lower():
        return "Literature indicates atypical angina combined with cholesterol > 240 increases risk."
    return "No significant recent literature flags for this symptom."

tools = [search_ehr_vector_db, search_clinical_literature]
llm = ChatGroq(model="llama-3.1-70b-versatile")
llm_with_tools = llm.bind_tools(tools)

# ==========================================
# 3. REACT AGENT & ROUTING (PHASE 1)
# ==========================================
def react_intake_agent(state: ClinicalState):
    sys_prompt = (
        "You are a clinical intake agent. Use tools to gather full patient history. "
        "Once you have gathered the data, summarize it and stop using tools."
    )
    messages = [{"role": "system", "content": sys_prompt}] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def route_react_agent(state: ClinicalState) -> Literal["tools", "supervisor_node"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "supervisor_node"

# ==========================================
# 4. SUPERVISOR NODE (PHASE 2)
# ==========================================
def supervisor_node(state: ClinicalState):
    sys_prompt = (
        "You are the Clinical Triage Supervisor. Review the conversation history.\n"
        "- If vitals are NOT cleanly extracted into a dictionary, route to 'validation_agent'.\n"
        "- If vitals are extracted but risk_score is 0.0, route to 'cardio_ml_worker'.\n"
        "- If risk_score is calculated, route to 'reporting_worker'."
    )
    messages = [{"role": "system", "content": sys_prompt}] + state["messages"]
    decision = llm.with_structured_output(SupervisorRouter).invoke(messages)
    return {"next_node": decision.next_node}

# ==========================================
# 5. DETERMINISTIC WORKERS
# ==========================================
def validation_agent(state: ClinicalState):
    """Parses text to extract a clean dictionary of features."""
    # In a real app, this would use a strict LLM extraction or Regex
    cleaned_vitals = {"age": 55, "chol": 245, "thalach": 140}
    return {
        "patient_vitals": cleaned_vitals, 
        "messages": [{"role": "assistant", "content": "Vitals strictly validated and formatted."}]
    }

def calculate_gaussian_probability(x: float, mean: float, stdev: float) -> float:
    """Mathematical implementation of the Gaussian PDF from scratch."""
    if stdev == 0.0:
        stdev = 1e-9
    exponent = math.exp(-((x - mean) ** 2 / (2 * stdev ** 2)))
    return (1 / (math.sqrt(2 * math.pi) * stdev)) * exponent

def cardio_ml_worker(state: ClinicalState):
    """Executes the classical ML prediction pipeline."""
    vitals = state.get("patient_vitals", {})
    age = vitals.get("age", 50)
    chol = vitals.get("chol", 200)

    # Class distribution parameters for "Heart Disease Positive"
    mean_age, std_age = 54.0, 9.0
    mean_chol, std_chol = 250.0, 40.0

    # Calculate Naive Bayes probabilities directly
    prob_age = calculate_gaussian_probability(age, mean_age, std_age)
    prob_chol = calculate_gaussian_probability(chol, mean_chol, std_chol)
    
    # Combined joint probability score (scaled)
    risk_score = round(prob_age * prob_chol * 1000, 4)
    
    return {
        "risk_score": risk_score, 
        "messages": [{"role": "assistant", "content": f"Gaussian ML risk score calculated: {risk_score}"}]
    }

def reporting_worker(state: ClinicalState):
    """Formats the final payload for the Streamlit UI."""
    vitals = state.get("patient_vitals", {})
    score = state.get("risk_score", 0.0)
    report = f"FINAL DIAGNOSTIC PAYLOAD:\n- Vitals: {vitals}\n- Calculated Gaussian Risk Score: {score}"
    return {"messages": [{"role": "assistant", "content": report}]}

# ==========================================
# 6. GRAPH ASSEMBLY
# ==========================================
builder = StateGraph(ClinicalState)

# Add all nodes
builder.add_node("react_intake", react_intake_agent)
builder.add_node("tools", ToolNode(tools))
builder.add_node("supervisor_node", supervisor_node)
builder.add_node("validation_agent", validation_agent)
builder.add_node("cardio_ml_worker", cardio_ml_worker)
builder.add_node("reporting_worker", reporting_worker)

# ReAct Loop Routing
builder.add_edge(START, "react_intake")
builder.add_conditional_edges("react_intake", route_react_agent)
builder.add_edge("tools", "react_intake")

# Supervisor Hierarchical Routing (Workers report back to Supervisor)
builder.add_edge("validation_agent", "supervisor_node")
builder.add_edge("cardio_ml_worker", "supervisor_node")
builder.add_edge("reporting_worker", "supervisor_node")

builder.add_conditional_edges(
    "supervisor_node",
    lambda state: state["next_node"],
    {
        "validation_agent": "validation_agent",
        "cardio_ml_worker": "cardio_ml_worker",
        "reporting_worker": "reporting_worker",
        "FINISH": END
    }
)

app = builder.compile()

# ==========================================
# 7. EXECUTION TRIGGER
# ==========================================
if __name__ == "__main__":
    initial_input = "Patient John Doe is experiencing chest pain."
    
    result = app.invoke({
        "messages": [HumanMessage(content=initial_input)],
        "patient_vitals": {},
        "risk_score": 0.0,
        "next_node": ""
    })
    
    for msg in result["messages"]:
        print(f"{msg.role.upper()}: {msg.content}\n")